# Improvised Enhanced Random Forest (ERF) Framework
### Ali Naqi (23F-3052) & Muhammad Ahmad (23F-3028) | SE-6B

This notebook demonstrates the replication and improvisation of the ERF framework for Apple App Store rating prediction. We apply **IQR Outlier Detection**, **PCA**, **Stratified K-Fold**, **RandomizedSearchCV**, and a **Soft Voting Ensemble** to achieve superior generalization on a large-scale dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

## 1. Data Analysis (EDA & IQR)
We load the dataset and remove statistical noise using the IQR method.

In [ ]:
df = pd.read_csv('appstore_7197_apps.csv')
df_filtered = df[df['Reviews'] >= 100].copy()

for col in ['Price', 'Size_Bytes']:
    Q1 = df_filtered[col].quantile(0.25)
    Q3 = df_filtered[col].quantile(0.75)
    IQR = Q3 - Q1
    df_filtered = df_filtered[(df_filtered[col] >= (Q1 - 1.5*IQR)) & (df_filtered[col] <= (Q3 + 1.5*IQR))]

print(f"Final clean dataset size: {len(df_filtered)} rows")

## 2. Feature Engineering & PCA
We engineer features and reduce them using PCA to retain 95% variance.

In [ ]:
df_filtered['Reviews_Per_Day'] = df_filtered['Reviews'] / 1000
df_filtered['Is_Free'] = (df_filtered['Price'] == 0).astype(int)

features = ['Size_Bytes', 'Price', 'Reviews', 'Is_Free']
X = df_filtered[features]
y = (df_filtered['Average_User_Rating'] >= 4.0).astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print(f"Reduced from {X.shape[1]} to {X_pca.shape[1]} Principal Components.")

## 3. Model Training (Stratified K-Fold)
Training all 3 base models plus our new **Soft Voting Ensemble**.

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
lr = LogisticRegression(max_iter=1000, C=0.5, solver='lbfgs')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
xgb = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=6, random_state=42)
ensemble = VotingClassifier(estimators=[('lr', lr), ('rf', rf), ('xgb', xgb)], voting='soft')

models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb, 'Ensemble (Soft Voting)': ensemble}
trained_models = {}
final_cm = None

for name, model in models.items():
    print(f"Training {name}...")
    scores = []
    for train_idx, test_idx in skf.split(X_pca, y):
        X_train, X_test = X_pca[train_idx], X_pca[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        scores.append(f1_score(y_test, y_pred))
        if name == 'Ensemble (Soft Voting)':
            final_cm = confusion_matrix(y_test, y_pred)
    trained_models[name] = np.mean(scores)
    print(f"  Average F1-Score: {np.mean(scores):.4f}\n")

## 4. Final Visualizations
Confusion Matrix for the top performing Ensemble model.

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(final_cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
plt.title("Confusion Matrix (Soft Voting Ensemble)")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()